# Multi-Query Attention — Solution

Reference implementation for `01_mqa_exercise.ipynb`.

## Setup

In [2]:
import torch
import torch.nn as nn

## Solution 1 — `MultiQueryAttention`

In [3]:
# 8 heads
# 64 - d_model

# q1,q2,q3,q4,q5,q6,q7,q8
# 8,8,8,8,8,8,8,8

# key vector - 8 shared by every head
# value vector - 8 shared by every head

In [5]:
(torch.randn(2,8,16,2) @ torch.randn(2,1,2,16)).shape

torch.Size([2, 8, 16, 16])

In [7]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, causal=False, max_seq_len=4096):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model//n_heads
        self.causal = causal

        self.qkv_projections = nn.Linear(d_model,d_model+2*self.d_head)
        self.out_projection = nn.Linear(d_model,d_model)

        self.register_buffer("causal_mask",torch.triu(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool),diagonal=1))

    def forward(self, x):
        bs,seq_len,_ = x.shape

        Q,K,V = torch.split(self.qkv_projections(x),[self.d_model,self.d_head,self.d_head],dim=-1)
        # Q -> bs,seq_len,d_model
        # K,V -> bs,seq_len,d_head
        Q = Q.view(bs,seq_len,self.n_heads,self.d_head).permute(0,2,1,3) #bs,n_heads,seq_len,d_head
        K = K.view(bs,seq_len,1,self.d_head).permute(0,2,1,3) #bs,1,seq_len,d_head
        V = V.view(bs,seq_len,1,self.d_head).permute(0,2,1,3) #bs,1,seq_len,d_head

        scores = Q@K.transpose(-2,-1)/self.d_head**0.5 #bs,n_heads,seq_len,seq_len
        
        if self.causal:
            scores = scores.masked_fill(self.causal_mask[:seq_len,:seq_len],float("-inf"))

        weights = torch.softmax(scores,dim=-1) #bs,n_heads,seq_len,seq_len
        #V -> bs,1,seq_len,d_head
        out = weights@V #bs,n_heads,seq_len,d_head
        out = out.permute(0,2,1,3).contiguous().view(bs,seq_len,self.d_model)

        out = self.out_projection(out)

        return out

## Solution 2 — Verify cache savings

In [8]:
# Compare W_k size: MQA vs equivalent MHA
d_model, n_heads = 512, 8
mqa = MultiQueryAttention(d_model=d_model, n_heads=n_heads)

# Inside qkv_projections, K's portion has output features = d_head
d_head = d_model // n_heads
mqa_kv_features = 2 * d_head                   # K + V together
mha_kv_features = 2 * d_model                  # K + V each have d_model output

print(f"d_model = {d_model}, n_heads = {n_heads}, d_head = {d_head}")
print(f"MHA  KV output features: {mha_kv_features}")
print(f"MQA  KV output features: {mqa_kv_features}")
print(f"Reduction: {mha_kv_features // mqa_kv_features}x  ({n_heads}x — one shared head instead of n_heads)")

# At inference, the KV cache size scales linearly with these:
seq_len = 8192
mha_cache = 2 * d_model * seq_len * 2          # 2 = bytes for fp16
mqa_cache = 2 * d_head  * seq_len * 2
print(f"\nKV cache at seq_len={seq_len} (fp16, 1 sample, 1 layer):")
print(f"  MHA: {mha_cache / 1e6:.1f} MB")
print(f"  MQA: {mqa_cache / 1e6:.1f} MB")

d_model = 512, n_heads = 8, d_head = 64
MHA  KV output features: 1024
MQA  KV output features: 128
Reduction: 8x  (8x — one shared head instead of n_heads)

KV cache at seq_len=8192 (fp16, 1 sample, 1 layer):
  MHA: 16.8 MB
  MQA: 2.1 MB


## Run the tests

In [9]:
from tests import run_mqa_tests
run_mqa_tests(MultiQueryAttention)

Running MultiQueryAttention...
  ✓ Output shape correct
  ✗ W_k is d_head-sized (not d_model-sized) — W_k params 5120 too large; should be O(d_head * d_model) ≈ 512, not d_model^2 = 4096
  ✓ Gradients flow
  ✓ Causal mode works

  3/4 checks passed — fix the ✗ above



False